# The leaves

**Week 1, Monday. Notebook 3 of 4.** By the end of this notebook you can count distinct customers,
state the first rate of the programme with its numerator, denominator and window, filter a loop
with an `if`, and predict three traps a real file sets before they spring.

> **The client asks.** "No averages. One business customer can move an average."
>
> Anand Iyer, finance controller, Kalpa Retail

The morning counted revenue. The tree says two more leaves are in the file, customers and orders
per customer, and this notebook counts both.

**Setup.** Find the helper, load the thirty orders, and recount revenue the way notebook 2 ended,
with `int()`, so this notebook runs on its own from a fresh kernel.

In [1]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

ORDERS = kit.load_records("C2_W01_D01_orders_STUDENT.py")
revenue = 0
for order in ORDERS:
    revenue += int(order["amount"])
print(len(ORDERS), "orders,", kit.rupees(revenue))

30 orders, Rs 5,44,810


In [2]:
kit.side_by_side(
    kit.ladder(["The question before the budget", "The first count", "The leaves", "The average that lies", "Hands-on: your call"], lit=2, show=False),
    kit.vflow(["customers\ndistinct people, not rows",
               "orders per customer\nthe first rate",
               "a filter\nan if inside the loop",
               "three traps\npredicted before they run",
               "the tree\nwith today's numbers on it"], show=False),
)

## Where the morning left the tree

| Leaf | Numerator over denominator | Where it stands |
|---|---|---|
| Revenue | The sum of amounts | Counted in notebook 2: Rs 5,44,810, all booked orders |
| Customers | Distinct customer ids | In the file, counted in this notebook |
| Orders per customer | Orders over distinct customers | In the file, counted in this notebook |
| Items per order, price per item | Items, and revenue over items | Need order lines, which this file does not carry |
| Discounts | Rupees given back | Need a discount field, which this file does not carry |

The solid leaves below are the two this notebook counts; the dashed ones wait for data the file
does not have.

In [3]:
kit.driver_tree({"label": "revenue", "note": "Rs 5,44,810 booked", "children": [
    {"label": "gross revenue", "note": "customers times spend", "children": [
        {"label": "customers", "note": "in the file", "kind": "known"},
        {"label": "revenue per customer", "note": "orders times order value", "children": [
            {"label": "orders per customer", "note": "in the file", "kind": "known"},
            {"label": "revenue per order", "note": "items times price", "children": [
                {"label": "items per order", "note": "not in this file", "kind": "unknown"},
                {"label": "price per item", "note": "not in this file", "kind": "unknown"}]}]}]},
    {"label": "discounts", "note": "not in this file", "kind": "unknown"}]},
    title="Two leaves this file can answer, and three it cannot")

## 1. Customers are people, not rows

Thirty rows are thirty orders, and some of them are the same person coming back. Counting customers
means counting each `customer_id` once: keep a list of ids already seen, and add an id only when
it is not in that list yet. Two new pieces arrive here, an `if` inside the loop and `not in`, which
checks every item of a list and reads like English.

**Predict before you run.** How many distinct customers are in thirty orders: 30, fewer, or more?

In [4]:
seen = []
steps = []
for order in ORDERS:
    new = order["customer_id"] not in seen
    if new:
        seen.append(order["customer_id"])
    if len(steps) < 8 or not new and len(steps) < 10:
        steps.append((order["order_id"], order["customer_id"],
                      "new, appended" if new else "already seen, skipped", len(seen)))
print(len(seen))

kit.table(["order", "customer_id", "what the if decided", "len(seen) after"], steps,
          caption="A desk check of the distinct count: the first eight orders, then the first two repeats")
kit.flow(["next order\nread its customer_id", "is it in seen?\nnot in reads like English",
          "no: append it\na new customer", "yes: skip it\na customer coming back"],
         lit=1, title="The distinct count: one question per order")

23


order,customer_id,what the if decided,len(seen) after
KR-01001,C-0101,"new, appended",1
KR-01002,C-0102,"new, appended",2
KR-01003,C-0103,"new, appended",3
KR-01004,C-0104,"new, appended",4
KR-01005,C-0105,"new, appended",5
KR-01006,C-0106,"new, appended",6
KR-01007,C-0107,"new, appended",7
KR-01008,C-0108,"new, appended",8
KR-01023,C-0101,"already seen, skipped",22
KR-01024,C-0102,"already seen, skipped",22


### Thirty orders, twenty-three customers: what are the other seven rows?

Choose before you read on: a) duplicates to delete before counting; b) customers who came back for
a second order; c) orders with a missing customer id; d) a mistake in the loop.

The answer is b, and a is the most useful wrong answer of the day, because a customer coming back
and a record entered twice look the same in a count. The cell below counts orders per customer and
checks the order ids, which is how the two are told apart.

In [5]:
orders_by_customer = {}
for order in ORDERS:
    cid = order["customer_id"]
    orders_by_customer[cid] = orders_by_customer.get(cid, 0) + 1

once = 0
twice = 0
for n in orders_by_customer.values():
    if n == 1:
        once += 1
    if n == 2:
        twice += 1

order_ids = []
for order in ORDERS:
    if order["order_id"] not in order_ids:
        order_ids.append(order["order_id"])

kit.stats([(len(seen), "customers", "distinct customer ids"),
           (once, "bought once", "one order each"),
           (twice, "came back", "two orders each"),
           (len(order_ids), "distinct order ids", "so no order is entered twice")])
kit.bars([("bought once", once), ("came back once", twice)],
         title="Customers by how many orders they placed in the window")

> **Kavya's review.** Check the order ids before you call anything a duplicate. Here all thirty
> are different, so the seven extra rows are real orders from customers who came back: the
> frequency branch at work.

The `.get(cid, 0) + 1` line is the accumulator again, one running total per customer, kept in a
dictionary. It is the same three moves with a dictionary as the total, which Week 0 used for the
librarian's days per department.

In [6]:
kit.check("twenty-three distinct customers", len(seen) == 23, f"got {len(seen)}")
kit.check("the customer counts add back to thirty orders", once + 2 * twice == len(ORDERS),
          f"{once} + 2 x {twice} = {once + 2 * twice}")
kit.check("every order id is different, so there are no duplicate rows",
          len(order_ids) == len(ORDERS), f"{len(order_ids)} distinct ids in {len(ORDERS)} rows")

## 2. Orders per customer, the first rate

A rate is a numerator over a denominator, and it leaves the team with its window. Division in
Python 3 always gives a float, so the full value prints; formatting it for a person is a separate
step that leaves the value alone.

**Predict before you run.** `round(30 / 23, 2)` and `f"{30 / 23:.2f}"`: do they print the same
thing?

In [7]:
orders = len(ORDERS)
customers = len(seen)
rate = orders / customers
print(rate)
print(round(rate, 2))
print(f"{rate:.2f}")

1.3043478260869565
1.3
1.30


**What happened.** `round()` gives back a number, and the number 1.30 is the number 1.3, so the
zero is dropped. The f-string formats for display and keeps two places. Both are correct; they do
different jobs.

> **Kavya's review.** A rate leaves the team with all three of its parts: "Orders per customer,
> 1 July to 26 September: 30 orders over 23 customers, 1.30." The average customer ordered 1.3
> times in the window.

In [8]:
kit.check("orders per customer is 1.30", f"{rate:.2f}" == "1.30", f"{rate:.4f}")
kit.check("round() returns a number, so the trailing zero goes", str(round(rate, 2)) == "1.3",
          str(round(rate, 2)))

## 3. A filter is an if inside the loop

Delivered revenue is the same accumulator, told which orders to skip. Two equals signs compare and
one assigns: `order["status"] == "delivered"` asks a question, and `delivered = 0` sets a value.

**Predict before you run.** Delivered revenue is lower than booked. By roughly how much: a few
hundred rupees, a few thousand, or a few lakh?

In [9]:
by_status = {"delivered": [0, 0], "returned": [0, 0], "cancelled": [0, 0]}
for order in ORDERS:
    row = by_status[order["status"]]
    row[0] += 1
    row[1] += int(order["amount"])

kit.table(["status", "orders", "amount"],
          [(s, n, kit.rupees(v)) for s, (n, v) in by_status.items()],
          caption="The same thirty orders, split by what happened to them")
kit.bars([(s, v) for s, (n, v) in by_status.items() if s != "delivered"],
         fmt=kit.rupees, title="What falls away between booked and delivered")

status,orders,amount
delivered,21,"Rs 5,20,790"
returned,5,"Rs 14,970"
cancelled,4,"Rs 9,050"


**What happened.** Delivered is Rs 5,20,790, a few thousand below booked on this file: Rs 14,970
of orders came back and Rs 9,050 were cancelled. The check a senior runs first is below: the parts
of a total must add up to the total.

In [10]:
parts = 0
for n, v in by_status.values():
    parts += v
kit.check("delivered, returned and cancelled add back to revenue", parts == revenue,
          f"{kit.rupees(parts)} against {kit.rupees(revenue)}")
kit.check("twenty-one orders were delivered", by_status["delivered"][0] == 21,
          str(by_status["delivered"][0]))

## 4. Predict three cells

Three one-liners, each a trap a real file sets. Write all three predictions before you run the
cell: a value, or the name of the error and why.

```python
ORDERS[0]["Amount"]          # cell a
"3500" > 3000                # cell b
round(30 / 23, 2)            # cell c
```

In [11]:
with kit.expect_error() as a:
    ORDERS[0]["Amount"]
with kit.expect_error() as b:
    "3500" > 3000
print(round(30 / 23, 2))

1.3


**What happened.** Two errors and a dropped zero.

| Cell | What happened | The habit it teaches |
|---|---|---|
| a | `KeyError: 'Amount'`: the key is `amount`, lower case, and a dictionary matches keys exactly | Copy key names from the record, never from memory |
| b | `TypeError: '>' not supported between instances of 'str' and 'int'`: Python will not decide whether text is bigger than a number | Know a field's type before you compare it |
| c | `1.3`: round() returned a number and the zero is display, never value | Format for people with an f-string |

Cell b is the morning's break in a new place. A loop that counts orders above Rs 3,000 stops on
the same record the sum stopped on, because `int()` in one loop fixed that loop only: the data
still holds text. That is the question Wednesday asks of the whole file.

In [12]:
kit.check("a key typed from memory raises KeyError", a.name == "KeyError", f"{a.name}: {a.message}")
kit.check("text never orders against a number", b.name == "TypeError" and "'>'" in b.message,
          f"{b.name}: {b.message}")

## 5. The tree, with today's numbers on it

Four numbers from one file, and they multiply back to the total: 23 customers times Rs 23,687 per
customer is Rs 5,44,810. Hold on to revenue per order, Rs 18,160. It is an average, and notebook 4
asks what it describes.

In [13]:
per_customer = revenue / customers
per_order = revenue / orders
kit.driver_tree({"label": "revenue", "note": kit.rupees(revenue), "children": [
    {"label": "customers", "note": str(customers), "kind": "known"},
    {"label": "revenue per customer", "note": kit.rupees(per_customer), "children": [
        {"label": "orders per customer", "note": f"{rate:.2f}", "kind": "known"},
        {"label": "revenue per order", "note": kit.rupees(per_order), "children": [
            {"label": "items per order", "note": "not in this file", "kind": "unknown"},
            {"label": "price per item", "note": "not in this file", "kind": "unknown"}]}]}]},
    title="Today's numbers on the tree; the discount branch waits for a field the file lacks")
kit.check("customers times revenue per customer comes back to revenue",
          round(customers * per_customer) == revenue, kit.rupees(customers * per_customer))
kit.check("orders per customer times revenue per order is revenue per customer",
          abs(rate * per_order - per_customer) < 0.01, f"{rate * per_order:,.2f}")

## 6. The three missing leaves, and how to ask for them

| Leaf | What it needs | The field to ask for |
|---|---|---|
| Items per order | One row per item sold, with its order | Order lines |
| Price per item | The price of each item before any discount | List prices |
| Discounts | What was given back, per order or per line | Discount in rupees |

> **Kavya's review.** Write the gap as a request, in the same note as the numbers: "Customers and
> frequency are counted. Basket, price and discounts need order lines, list prices and discounts,
> which this file does not carry."

In [14]:
kit.table(["What this notebook established", "The evidence"],
          [("Customers are distinct ids, counted once each", f"{customers} customers in {orders} orders"),
           ("A repeat order is different from a duplicate row", f"{twice} came back; 30 distinct order ids"),
           ("A rate carries numerator, denominator and window", "30 over 23, 1 July to 26 September: 1.30"),
           ("A filter is an if inside the accumulator", "delivered Rs 5,20,790; the parts add back"),
           ("Three traps: exact keys, text against numbers, round against format",
            "KeyError, TypeError, 1.3")],
          caption="Summary: the leaves")
kit.check_summary()
print("Next: notebook 4 asks whether Rs 18,160 describes a typical order, and why Anand said no averages.")

What this notebook established,The evidence
"Customers are distinct ids, counted once each",23 customers in 30 orders
A repeat order is different from a duplicate row,7 came back; 30 distinct order ids
"A rate carries numerator, denominator and window","30 over 23, 1 July to 26 September: 1.30"
A filter is an if inside the accumulator,"delivered Rs 5,20,790; the parts add back"
"Three traps: exact keys, text against numbers, round against format","KeyError, TypeError, 1.3"


Next: notebook 4 asks whether Rs 18,160 describes a typical order, and why Anand said no averages.
